In [ ]:
import sys
from pathlib import Path

ROOT = Paththr.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from imdb_spoiler_io import load_raw_imdb_spoiler_json, prepare_reviews_dataframe, save_processed_reviews
from splitters import SplitConfig, split_by_movie_id
from head_tail import apply_head_tail_truncation

paths = get_paths(ROOT)
print(f"Data Raw Path: {paths.data_raw}")
print(f"Data Processed Path: {paths.data_processed}")

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data Raw Path: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\raw
Data Processed Path: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed


In [ ]:
MODEL_NAME = "bert-base-uncased"
SEED = 42

TRAIN_SIZE = 0.80
VAL_SIZE = 0.1
TEST_SIZE = 0.1

MAX_LEN = 512
HEAD_LEN = 128

SAMPLE_SIZE = 300000

In [ ]:
print("Loading raw data...")
df_raw = load_raw_imdb_spoiler_json(paths.data_raw)

df_all, schema = prepare_reviews_dataframe(df_raw)
print(f"Totale recensioni caricate: {len(df_all)}")
display(df_all.head())

Loading raw data...
Loading specific file: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\raw\IMDB_reviews.json
Totale recensioni caricate: 573913


,review_id,movie_id,text,label
0,0,tt0111161,"In its Oscar year, Shawshank Redemption (writt...",1
1,1,tt0111161,The Shawshank Redemption is without a doubt on...,1
2,2,tt0111161,I believe that this film is the best story eve...,1
3,3,tt0111161,"**Yes, there are SPOILERS here**This film has ...",1
4,4,tt0111161,At the heart of this extraordinary movie is a ...,1


In [ ]:
import numpy as np

if SAMPLE_SIZE and SAMPLE_SIZE < len(df_all):
    print(f"Subsampling to {SAMPLE_SIZE} reviews (keeping movie groups)...")
    unique_movies = df_all['movie_id'].unique()

    selected_movies = np.random.choice(unique_movies, size=int(len(unique_movies) * (SAMPLE_SIZE/len(df_all))), replace=False)
    df_all = df_all[df_all['movie_id'].isin(selected_movies)].copy()
    print(f"New size: {len(df_all)}")

print("Splitting dataset grouped by movie_id...")
cfg = SplitConfig(train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE, seed=SEED)
train_df, val_df, test_df = split_by_movie_id(df_all, cfg=cfg)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Subsampling to 300000 reviews (keeping movie groups)...
New size: 297679
Splitting dataset grouped by movie_id...
Train: 231423 | Val: 34912 | Test: 31344


In [ ]:
print(f"Tokenizing with {MODEL_NAME} using Head+Tail strategy...")

train_tokenized = apply_head_tail_truncation(train_df, MODEL_NAME, max_length=MAX_LEN, head_len=HEAD_LEN)
val_tokenized = apply_head_tail_truncation(val_df, MODEL_NAME, max_length=MAX_LEN, head_len=HEAD_LEN)
test_tokenized = apply_head_tail_truncation(test_df, MODEL_NAME, max_length=MAX_LEN, head_len=HEAD_LEN)

print("Tokenization complete.")
display(train_tokenized[['input_ids', 'attention_mask']].head(2))

Tokenizing with bert-base-uncased using Head+Tail strategy...
Strategy: [CLS] + Head(128) + Tail(382) + [SEP] = 512


Applying Head+Tail: 100%|██████████| 231423/231423 [00:04<00:00, 57316.01it/s] 


Strategy: [CLS] + Head(128) + Tail(382) + [SEP] = 512


Applying Head+Tail: 100%|██████████| 34912/34912 [00:00<00:00, 151342.14it/s]


Strategy: [CLS] + Head(128) + Tail(382) + [SEP] = 512


Applying Head+Tail: 100%|██████████| 31344/31344 [00:01<00:00, 20884.12it/s] 


Tokenization complete.


,input_ids,attention_mask
0,"[101, 1000, 2045, 2064, 2022, 2053, 4824, 2090...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,"[101, 4557, 4811, 8872, 18155, 2050, 2320, 205...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
out_dir = paths.data_processed
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving to {out_dir}...")

train_tokenized.to_parquet(out_dir / "train.parquet", index=False)
val_tokenized.to_parquet(out_dir / "val.parquet", index=False)
test_tokenized.to_parquet(out_dir / "test.parquet", index=False)

print("Done! Ready for training.")

Saving to C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\tokenized...
Done! Ready for training.
